# FADE Quickstart

**Frequency-Adaptive Decay Encoding** — drop-in KV cache compression for HuggingFace transformers.

This notebook demonstrates FADE's presets and the rotated 2-bit backend (6.2× compression) on Qwen2.5-0.5B-Instruct.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omodaka9375/fade/blob/main/examples/quickstart.ipynb)

In [ ]:
# Install FADE (run once)
!pip install -q fade-kv

In [ ]:
import torch
from transformers import DynamicCache
from fade import FadeConfig, create_tiered_cache
from fade.backends import get_backend
from fade.eval.memory import cache_storage_bytes
from fade.patch import forward_with_tracking, load_model
from fade.policy import reassign_tiers, reassign_tiers_by_position
from fade.tracker import AttentionTracker

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
PROMPT = (
    "Explain the following topics in detail, one paragraph each:\n"
    "1. How CPU cache hierarchies work\n"
    "2. The history of the Roman road network\n"
    "3. How photosynthesis converts sunlight to energy\n"
    "4. The basics of quantum computing and qubits\n"
)
MAX_NEW = 128
REASSIGN_EVERY = 32

print(f"Device: {DEVICE}")
model, tokenizer = load_model(MODEL_ID, device_map=DEVICE, dtype=DTYPE, attn_impl="eager")
num_layers = model.config.num_hidden_layers
head_dim = model.config.hidden_size // model.config.num_attention_heads
print(f"Model loaded: {MODEL_ID}, {num_layers} layers, head_dim={head_dim}")

In [ ]:
def run_tiered(label, config, backend=None):
    """Run decode with tier reassignment and report KV cache size."""
    enc = tokenizer(PROMPT, return_tensors="pt").to(DEVICE)
    kwargs = {"config": config}
    if backend is not None:
        kwargs["quant_backend"] = backend
    cache = create_tiered_cache(model, dtype=DTYPE, **kwargs)
    tracker = AttentionTracker(num_layers=num_layers)
    out = forward_with_tracking(model, enc.input_ids, cache, tracker=tracker)
    next_tok = out.logits[:, -1:, :].argmax(dim=-1)
    for step in range(MAX_NEW - 1):
        out = forward_with_tracking(model, next_tok, cache, tracker=tracker)
        next_tok = out.logits[:, -1:, :].argmax(dim=-1)
        if (step + 1) % REASSIGN_EVERY == 0:
            reassign_tiers_by_position(cache, num_layers)
        if tokenizer.eos_token_id is not None and next_tok.item() == tokenizer.eos_token_id:
            break
    kv = cache.compressed_storage_bytes() / (1024 * 1024)
    return kv

In [ ]:
# Baseline: plain FP16 DynamicCache
enc = tokenizer(PROMPT, return_tensors="pt").to(DEVICE)
base_cache = DynamicCache()
with torch.no_grad():
    model.generate(**enc, past_key_values=base_cache, max_new_tokens=MAX_NEW, do_sample=False)
base_kv = cache_storage_bytes(base_cache) / (1024 * 1024)
print(f"Baseline FP16: {base_kv:.2f} MiB")

In [ ]:
# Safe preset (INT4, no eviction, ~3.5x)
safe_kv = run_tiered("Safe", FadeConfig.safe())
print(f"Safe INT4:      {safe_kv:.2f} MiB  ({base_kv/safe_kv:.1f}x)")

In [ ]:
# Balanced preset (INT4 + eviction, ~5x)
bal_kv = run_tiered("Balanced", FadeConfig.balanced().with_overrides(eviction_policy="position"))
print(f"Balanced:       {bal_kv:.2f} MiB  ({base_kv/bal_kv:.1f}x)")

In [ ]:
# Rotated 2-bit (~6.2x compression)
rot2_backend = get_backend("rotated", head_dim=head_dim, bits=2)
rot2_kv = run_tiered("Rotated 2-bit", FadeConfig.safe(), backend=rot2_backend)
print(f"Rotated 2-bit:  {rot2_kv:.2f} MiB  ({base_kv/rot2_kv:.1f}x)")

In [ ]:
print(f"\n{'='*50}")
print(f"  Summary")
print(f"{'='*50}")
print(f"  Baseline FP16: {base_kv:.2f} MiB")
print(f"  Safe INT4:     {safe_kv:.2f} MiB  ({base_kv/safe_kv:.1f}x)")
print(f"  Balanced:      {bal_kv:.2f} MiB  ({base_kv/bal_kv:.1f}x)")
print(f"  Rotated 2-bit: {rot2_kv:.2f} MiB  ({base_kv/rot2_kv:.1f}x)")